In [ ]:
# ------------------------------
# Text preprocessing utils
# ------------------------------
from __future__ import annotations

import html
import re
import unicodedata

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# If stopwords are not present locally, download once per environment.
nltk.download("stopwords", quiet=True)

RU_STOPWORDS = frozenset(stopwords.words("russian"))
EN_STOPWORDS = frozenset(stopwords.words("english"))

STEMMER_RU = SnowballStemmer("russian")
STEMMER_EN = SnowballStemmer("english")

TECH_TOKENS = frozenset({
    "python", "sql", "pytorch", "tensorflow", "keras", "sklearn",
    "numpy", "pandas", "spark", "pyspark", "airflow", "docker",
    "kubernetes", "mlops", "nlp", "llm", "rag", "opencv", "xgboost",
    "lightgbm", "catboost", "fastapi", "flask", "grpc", "api",
    "aws", "gcp", "azure", "linux", "bash", "git", "etl", "elt",
    "gpu", "cuda", "onnx", "tensorrt", "bert", "gpt", "transformer",
    "ml", "dl", "java", "scala", "go", "golang", "cplusplus", "csharp", "dotnet",
    "machine_learning", "deep_learning", "computer_vision",
    "natural_language_processing", "retrieval_augmented_generation",
    "team_lead", "tech_lead",
})

# Precompiled regex patterns (speed + readability)
_RE_URL = re.compile(r"http\S+|www\.\S+", flags=re.IGNORECASE)
_RE_EMAIL = re.compile(r"\S+@\S+", flags=re.IGNORECASE)
_RE_DIGIT_PLUS = re.compile(r"(\d)\+")
_RE_NON_WORD = re.compile(r"[^a-zа-яё0-9_\s]", flags=re.IGNORECASE)
_RE_SPACES = re.compile(r"\s+")
_RE_RU_WORD = re.compile(r"[а-яё]+")
_RE_EN_WORD = re.compile(r"[a-z_]+")

# Prefer longer multi-word replacements first
_REPLACEMENTS: tuple[tuple[str, str], ...] = (
    ("retrieval-augmented generation", "retrieval_augmented_generation"),
    ("retrieval augmented generation", "retrieval_augmented_generation"),
    ("natural language processing", "natural_language_processing"),
    ("large language models", "large_language_models"),
    ("large language model", "large_language_model"),
    ("machine learning engineer", "machine_learning_engineer"),
    ("data scientist", "data_scientist"),
    ("research scientist", "research_scientist"),
    ("deep learning", "deep_learning"),
    ("machine learning", "machine_learning"),
    ("computer vision", "computer_vision"),
    ("tech lead", "tech_lead"),
    ("team lead", "team_lead"),
    ("ci/cd", "ci_cd"),
    ("c++", "cplusplus"),
    ("c#", "csharp"),
    (".net", "dotnet"),
)


def normalize_text(text: object) -> str:
    """Normalize raw vacancy text into a token-friendly string.

    Steps:
    - handle NaN/None
    - unescape html entities
    - unicode normalize + lower
    - remove urls/emails
    - replace common multiword tech terms with underscore versions
    - keep only letters/digits/underscore/space
    """
    if text is None or (isinstance(text, float) and pd.isna(text)) or pd.isna(text):
        return ""

    s = unicodedata.normalize("NFKC", html.unescape(str(text))).lower()
    s = _RE_URL.sub(" ", s)
    s = _RE_EMAIL.sub(" ", s)
    for old, new in _REPLACEMENTS:
        s = s.replace(old, new)
    s = _RE_DIGIT_PLUS.sub(r"\1plus", s)
    s = _RE_NON_WORD.sub(" ", s)
    s = _RE_SPACES.sub(" ", s).strip()
    return s


def _is_russian_word(token: str) -> bool:
    return bool(_RE_RU_WORD.fullmatch(token))


def _is_english_word(token: str) -> bool:
    return bool(_RE_EN_WORD.fullmatch(token))


def stem_token(token: str) -> str:
    """Stem a token unless it is a technical token (or contains underscore)."""
    if token in TECH_TOKENS or "_" in token:
        return token
    if _is_russian_word(token):
        return STEMMER_RU.stem(token)
    if _is_english_word(token):
        return STEMMER_EN.stem(token)
    return token


def preprocess_for_tfidf(text: object) -> str:
    """Full preprocessing pipeline for TF-IDF input."""
    tokens = normalize_text(text).split()
    out: list[str] = []
    for t in tokens:
        if len(t) < 2 and t not in TECH_TOKENS:
            continue
        if (t in RU_STOPWORDS or t in EN_STOPWORDS) and t not in TECH_TOKENS:
            continue
        out.append(stem_token(t))
    return " ".join(out)

[nltk_data] Downloading package stopwords to /home/kali/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Apply preprocessing
if "df" not in globals():
    raise NameError("`df` is not defined. Load your dataset into a pandas DataFrame named `df` first.")

if "vacancy_text" not in df.columns:
    raise KeyError("Column `vacancy_text` not found in df. Available columns: " + ", ".join(df.columns.astype(str)))

df = df.copy()
df["text_preprocessed"] = df["vacancy_text"].apply(preprocess_for_tfidf)
df = df[df["text_preprocessed"].str.strip().ne("")].copy()

print("Rows after preprocessing:", len(df))